1. Імпорт бібліотек та завантаження даних:

(Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;
Приклад посилання для області №8:
https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID=8&year1=1981&year2=2024&type=Mean
Якщо вказати provinceID=0, сформується вибірка в середньому по Україні. Її завантажувати не потрібно;)

In [3]:
import os
import urllib.request
import datetime
import pandas as pd
import glob

os.makedirs('./lab_02/data', exist_ok=True)

def download_noaa_data():
    base_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={}&year1=1981&year2=2024&type=Mean"
    
    for province_id in range(1, 28):
        existing_files = glob.glob(f'./lab_02/data/vhi_id_{province_id}_*.csv')
        if existing_files:
            print(f"Файл для області {province_id} вже існує. Пропускаємо.")
            continue
            
        url = base_url.format(province_id)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"./lab_02/data/vhi_id_{province_id}_{timestamp}.csv"
        
        try:
            with urllib.request.urlopen(url) as response:
                html = response.read().decode('utf-8')
                
            lines = html.split('\n')
            clean_data = []
            for line in lines:
                if not line.startswith("<") and not line.startswith("</") and line.strip() != "":
                    clean_data.append(line.replace(', ', ','))
            
            with open(filename, 'w') as f:
                f.write('\n'.join(clean_data))
            print(f"Завантажено: {filename}")
        except Exception as e:
            print(f"Помилка завантаження {province_id}: {e}")

download_noaa_data()

Завантажено: ./lab_02/data/vhi_id_1_20260314_140148.csv
Завантажено: ./lab_02/data/vhi_id_2_20260314_140150.csv
Завантажено: ./lab_02/data/vhi_id_3_20260314_140151.csv
Завантажено: ./lab_02/data/vhi_id_4_20260314_140152.csv
Завантажено: ./lab_02/data/vhi_id_5_20260314_140153.csv
Завантажено: ./lab_02/data/vhi_id_6_20260314_140154.csv
Завантажено: ./lab_02/data/vhi_id_7_20260314_140155.csv
Завантажено: ./lab_02/data/vhi_id_8_20260314_140156.csv
Завантажено: ./lab_02/data/vhi_id_9_20260314_140157.csv
Завантажено: ./lab_02/data/vhi_id_10_20260314_140158.csv
Завантажено: ./lab_02/data/vhi_id_11_20260314_140159.csv
Завантажено: ./lab_02/data/vhi_id_12_20260314_140200.csv
Завантажено: ./lab_02/data/vhi_id_13_20260314_140201.csv
Завантажено: ./lab_02/data/vhi_id_14_20260314_140201.csv
Завантажено: ./lab_02/data/vhi_id_15_20260314_140202.csv
Завантажено: ./lab_02/data/vhi_id_16_20260314_140203.csv
Завантажено: ./lab_02/data/vhi_id_17_20260314_140204.csv
Завантажено: ./lab_02/data/vhi_id_18_202

2. Зчитування, очищення та заміна індексів:

(Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області (див. How to Handle Missing Data in Python?)
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). )

In [4]:
def prepare_dataframe():
    files = glob.glob('./lab_02/data/vhi_id_*.csv')
    df_list = []
    
    noaa_to_ua = {
        1: 22,
        2: 24,
        3: 23,
        4: 25,
        5: 3,
        6: 4,
        7: 8,
        8: 19,
        9: 20,
        10: 21,
        11: 9,
        12: 26,
        13: 10,
        14: 11,
        15: 12,
        16: 13,
        17: 14,
        18: 15,
        19: 16,
        20: 27,
        21: 17,
        22: 18,
        23: 6,
        24: 1,
        25: 2,
        26: 7,
        27: 5
    }
    
    province_names = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
        6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
        11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
        16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'Республіка Крим',
        26: 'м. Київ', 27: 'м. Севастополь'
    }

    for file in files:

        old_id = int(file.split('vhi_id_')[1].split('_')[0])
        
        df = pd.read_csv(file, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty'])
        df = df.drop(columns=['empty'], errors='ignore')
        df = df.dropna()
        
        df['Year'] = df['Year'].astype(str).str.replace('<tt><pre>', '').astype(int)
        
        df = df[df['VHI'] != -1.0]
        
        new_id = noaa_to_ua.get(old_id, old_id)
        df['Province_ID'] = new_id
        df['Province_Name'] = province_names.get(new_id, 'Unknown')
        
        df_list.append(df)
        
    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

df = prepare_dataframe()
df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,2,0.054,262.29,46.83,31.75,39.29,22,Черкаська
1,1982,3,0.055,263.82,48.13,27.24,37.68,22,Черкаська
2,1982,4,0.053,265.33,46.09,23.91,35.00,22,Черкаська
3,1982,5,0.050,265.66,41.46,26.65,34.06,22,Черкаська
4,1982,6,0.048,266.55,36.56,29.46,33.01,22,Черкаська


3. Процедури формування вибірок:

(Реалізувати процедури для формування вибірок (https://www.geeksforgeeks.org/pandas/ways-to-filter-pandas-dataframe-by-column-values/)  наступного виду:
Ряд VHI для області за вказаний рік;
Ряд VHI за вказаний діапазон років для вказаних областей;
Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани;
Інші вибірки на вимогу викладача.)

In [5]:
def get_vhi_by_year(df, province_id, year):
    return df[(df['Province_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]

def get_vhi_range(df, province_ids, year_start, year_end):
    return df[(df['Province_ID'].isin(province_ids)) & 
              (df['Year'] >= year_start) & 
              (df['Year'] <= year_end)][['Year', 'Week', 'Province_Name', 'VHI']]

def get_extremes(df, province_id, year):
    subset = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    vhi_series = subset['VHI']
    
    return {
        'Min': vhi_series.min(),
        'Max': vhi_series.max(),
        'Mean': vhi_series.mean(),
        'Median': vhi_series.median()
    }

print("VHI для Вінницької області (ID=1) за 2020 рік:")
print(get_vhi_by_year(df, 1, 2020).head())

print("\nЕкстремуми для Вінницької області за 2020 рік:")
print(get_extremes(df, 1, 2020))

VHI для Вінницької області (ID=1) за 2020 рік:
       Week    VHI
52180     1  40.92
52181     2  43.19
52182     3  44.74
52183     4  45.29
52184     5  44.80

Екстремуми для Вінницької області за 2020 рік:
{'Min': np.float64(34.48), 'Max': np.float64(64.12), 'Mean': np.float64(45.911538461538456), 'Median': np.float64(44.230000000000004)}
